In [ ]:
# import libraries
import pandas as pd
import plotly.graph_objects as go

In [ ]:
# load data
df_prod = pd.read_csv('../data/Global_production_quantity.csv')
df_species = pd.read_csv('../data/CL_FI_SPECIES_GROUPS.csv')
df_countries = pd.read_csv('../data/CL_FI_COUNTRY_GROUPS.csv')

In [ ]:
# preview production data
df_prod.info()

In [ ]:
# preview specie data
df_species.head()

In [ ]:
# preview country data
df_countries.head()

In [ ]:
# filter specie data to keep only seaweed species
mask_columns = ["CPC_Class_Es","CPC_Class_Ar","CPC_Class_Cn","CPC_Class_Ru", "ISSCAAP_Group_Cn", "ISSCAAP_Group_Ru", "CPC_Class_Fr","CPC_Group_Fr","CPC_Group_Es", "CPC_Group_Ar","CPC_Group_Cn","CPC_Group_Ru", "ISSCAAP_Group_Es","ISSCAAP_Group_Ar", "Yearbook_Group_Es", "Yearbook_Group_Ar", "Yearbook_Group_Cn", "Yearbook_Group_Ru", "ISSCAAP_Group_En"]
df_species_filter_alguae = df_species.drop(columns=mask_columns)[df_species["ISSCAAP_Group_Fr"].isin(["Algues rouges", "Algues brunes", "Algues vertes"])]

# get list of seaweed species
list_algae_species = df_species_filter_alguae["3A_Code"].tolist()

# filter production data to keep only seaweed species
df_prod_alguae = df_prod[df_prod["SPECIES.ALPHA_3_CODE"].isin(list_algae_species)]

# rename some columns
df_prod_alguae = df_prod_alguae.rename(columns={"PRODUCTION_SOURCE_DET.CODE": "source_production","COUNTRY.UN_CODE": "UN_Code", "PERIOD": "Année","VALUE" : "Production"})

# rename production origin (as Récolte or Culture)
df_prod_alguae["source_production"] = df_prod_alguae["source_production"].apply(lambda x: "Récolte" if x == "CAPTURE" else "Culture")


In [ ]:
# filter country data to keep relevant columns only
df_countries_filter = df_countries[["UN_Code", 'Name_Fr', "Continent_Group_Fr"]]

# fix French name of South Korea
mask_korea = df_countries_filter["Name_Fr"] == "République de Corée"
df_countries_filter.loc[mask_korea, "Name_Fr"] = "Corée du Sud"

In [ ]:
# merge production and country data
df_prod_alguae_country = df_prod_alguae.merge(df_countries_filter, on="UN_Code", how="left")

## Plot European harvesting versus culture proportions

In [ ]:
# copy data to plot
data_fig4 = df_prod_alguae_country.copy()

# rename some columns
data_fig4 = data_fig4.rename(columns={"Name_Fr": "Pays", "Continent_Group_Fr":"Continent"})

# select european countries
mask_eu = data_fig4["Continent"] == "Europe"
data_fig4 = data_fig4[mask_eu]

# group data by type of production
data_fig4 = data_fig4[["source_production", "Production"]].groupby("source_production").sum().reset_index()

### PC version

In [ ]:
# set colors
colors= ["#0950AD","#5B8FCB" ]

# plot proportions
fig4 = go.Figure()

fig4.add_trace(
    go.Pie(
        labels=data_fig4["source_production"],
        values=data_fig4["Production"],
        pull=[0.2, 0],
        texttemplate = "%{label} <br> %{percent:.0%}"
    ))

fig4.update_traces(hoverinfo='skip', textfont_size=14,
                  marker=dict(colors=colors), textposition='outside')
fig4.update_layout(
    paper_bgcolor ="#F9BF6B",
    plot_bgcolor ="#F9BF6B",
    font_family="Montserrat",
    font_color="#1D3B6E",
    showlegend=False,
    margin=dict(
        l=10,
        r=10,
        t=10,
        b=10
    ),
    height=400,
    width=400,
)

fig4.show()

In [ ]:
fig4.write_html("../figures/piechart_europe_pc.html", include_plotlyjs="cdn")

### Mobile version

In [ ]:
# set colors
colors= ["#0950AD","#5B8FCB" ]

# plot proportions
fig4 = go.Figure()

fig4.add_trace(
    go.Pie(
        labels=data_fig4["source_production"],
        values=data_fig4["Production"],
        pull=[0.2, 0],
        texttemplate = "%{label} <br> %{percent:.0%}"
    ))

fig4.update_traces(hoverinfo='skip', textfont_size=14,
                  marker=dict(colors=colors), textposition='outside')
fig4.update_layout(
    paper_bgcolor ="#F9BF6B",
    plot_bgcolor ="#F9BF6B",
    font_family="Montserrat",
    font_color="#1D3B6E",
    showlegend=False,
    margin=dict(
        l=5,
        r=5,
        t=5,
        b=5
    ),
    height=250,
    width=250,
)

fig4.show()

In [ ]:
fig4.write_html("../figures/piechart_europe_mobile.html", include_plotlyjs="cdn")